# Stage 2 · 整轨迹 DPO + 核验上下文双视图
干净状态 vs 追加主题相关但不提供支持的 context。L_task(two views) + 0.2 DPO + 0.1 stability；β=0.1，容忍区间 0.05。预算耗尽不算自主 STOP。

In [1]:
from pathlib import Path
import os, sys, subprocess, zipfile
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/SA-Jev')
    PROJECT = Path('/content/graphrag-jev')
except ImportError:
    DRIVE = Path(os.environ.get('SAJEV_WORKDIR', '../outputs/colab_local')).resolve()
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DRIVE.mkdir(parents=True, exist_ok=True)
REPO_URL = os.environ.get('SAJEV_REPO_URL', 'https://github.com/BoldHu/SA-Jev.git')  # Override with SAJEV_REPO_URL if needed.
SYNC_GIT = True
if not PROJECT.exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
    elif (DRIVE / 'sa-jev-source.zip').exists():
        PROJECT.mkdir(parents=True)
        with zipfile.ZipFile(DRIVE / 'sa-jev-source.zip') as archive:
            archive.extractall(PROJECT)
    else:
        raise RuntimeError('Set SAJEV_REPO_URL or copy sa-jev-source.zip to MyDrive/SA-Jev/')
if SYNC_GIT and (PROJECT / '.git').exists():
    status = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True)
    if status.strip():
        print('Local edits present; preserving them. Git synchronization skipped.')
    else:
        subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
MODE = 'full'  # Change to 'full' only after the pilot completes and memory is measured.
ROOT = DRIVE / 'public_lightrag_v1' / MODE
ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(DRIVE / 'huggingface')
PUBLIC_ASSETS = DRIVE / 'public_lightrag_assets'
CONFIG_PATH = PROJECT / 'configs/public_lightrag.yaml'
print('Code:', PROJECT, 'Artifacts:', ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Code: /content/graphrag-jev Artifacts: /content/drive/MyDrive/SA-Jev/public_lightrag_v1/full


In [2]:
# Run this in each fresh Colab runtime. No model weights are downloaded here.
subprocess.run([sys.executable, 'scripts/bootstrap_upstream.py'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[train,retrieval,test]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', 'vendor/OpenJev'], check=True)
if 'google.colab' in sys.modules:
    # BF16 + LoRA does not use torchao quantization. Colab's old optional
    # torchao package makes PEFT 0.19.1 fail even on ordinary Linear layers.
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'fla-core==0.5.2'], check=True)
# Activate editable-install .pth files in this already-running notebook kernel.
import importlib, site
site_dirs = site.getsitepackages()
if site.ENABLE_USER_SITE:
    site_dirs.append(site.getusersitepackages())
for site_dir in dict.fromkeys(site_dirs):
    site.addsitedir(site_dir)
importlib.invalidate_caches()
import sa_jev
print('SA-Jev source:', sa_jev.__file__)
# If package installation asks for a runtime restart, restart before importing torch.
from sa_jev.experiments import load_config, environment_manifest
config = load_config(CONFIG_PATH)
environment_manifest(ROOT / 'environment.json', config)
def cli(command, *args):
    from collections import deque
    tail = deque(maxlen=40)
    cmd = [sys.executable, '-u', '-m', 'sa_jev.cli', command, '--root', str(ROOT), '--config', str(CONFIG_PATH), *map(str,args)]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                tail.append(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if returncode:
        raise RuntimeError(f'sajev {command} failed (exit {returncode}). Last output: ' + ''.join(tail))


SA-Jev source: /content/graphrag-jev/src/sa_jev/__init__.py


In [3]:
# Read the same credential file in every notebook. Fill it in once.
from sa_jev.environment import load_environment
ENV_FILE = (DRIVE if 'google.colab' in sys.modules else PROJECT) / '.env'
credential_status = load_environment(ENV_FILE, template=PROJECT / '.env.example')
print('Credential file:', ENV_FILE)
for name in ('OPENAI_API_KEY', 'OPENAI_BASE_URL', 'TYPESAFE_API_KEY', 'HF_TOKEN'):
    print(name + ':', 'loaded' if credential_status[name] else 'empty')
if not credential_status['OPENAI_API_KEY']:
    print('Fill OPENAI_API_KEY in the file above, save it, then rerun this cell before building the index or answering questions.')


Credential file: /content/drive/MyDrive/SA-Jev/.env
OPENAI_API_KEY: loaded
OPENAI_BASE_URL: loaded
TYPESAFE_API_KEY: loaded
HF_TOKEN: loaded


## 40GB A100：Stage 2 优化路径测试
MODE='full'、RUN_FULL_STAGE2=False。使用 02 最近成功的 probe 检查点，最多审核 16 道训练题、每题 2 段（最多 32 次未缓存审核，网络重试另计），最多 8 对真实完整轨迹。
选题保留此前最先审核的 4 题，并加入大 K／长证据样本及固定随机样本。不是用于报告模型效果的测试集。
默认 8 步、有效批量 8；若小样本不足会明确重复若干 epoch。审核失败的题仍做干净任务，不强制稳定性。
输出写入独立 stage2_probe；不覆盖正式上下文包、参考概率和模型。先完成 02 的两种精度检查；新运行时也可运行下面的小模型检查。

In [4]:
# Small Qwen3.5 + LoRA check: no downloads, no API, no 4B training.
import torch
assert torch.cuda.is_available(), 'Select an A100 GPU runtime'
assert torch.cuda.is_bf16_supported(including_emulation=False), 'Native BF16 GPU required'
for check_dtype in ('float32', 'bfloat16'):
    log_path = ROOT / f'qwen_execution_{check_dtype}.log'
    cmd = [sys.executable, '-u', 'scripts/check_qwen_lora.py', '--device', 'cuda',
           '--compare-execution', '--dtype', check_dtype,
           '--output', str(ROOT/f'qwen_execution_{check_dtype}.json')]
    with log_path.open('w', encoding='utf-8') as log:
        with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True); log.write(line); log.flush()
                status = process.wait()
            except KeyboardInterrupt:
                process.terminate(); process.wait(); raise
    if status:
        raise RuntimeError(f'Execution equivalence check failed; inspect {log_path}. Do not start 4B training.')


[ERROR] `loss` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen3_5/modeling_qwen3_5.py.
[ERROR] `logits` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen3_5/modeling_qwen3_5.py.
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
[data] Build trajectory preferences: starting 0/1; waiting for records...
[execution check] {'optimized': False, 'checkpoint_mode': 'branch', 'seconds': 3.7931864749989472, 'max_gradient_abs_error': 0.0, 'relative_gradient_l2_error': 0.0, 'execut

## 设置（在这里改）
40GB 先保持 PROFILE=a100_40gb。RUN_FULL_STAGE2=True 会启动全量审核及正式训练，并要求正式 Stage 1 检查点；现在保持 False。

In [5]:
import copy, importlib, torch
from datetime import datetime, timezone
from uuid import uuid4
from sa_jev import execution, model, trajectories, training, performance, experiments, context_views, probes
for module in (execution, model, trajectories, training, performance, experiments, context_views, probes):
    importlib.reload(module)
RUN_FULL_STAGE2 = False
PROBE_STEPS = 8
PROBE_QUESTIONS = 16
PROBE_PAIRS = 8
TRAIN_SEEDS = [17]
PROFILE = 'a100_40gb'  # Later: 'a100_80gb' or 'h100_80gb'.
CHECKPOINT_MODE = 'branch'
STRESS_CHECK = True
STAGE1_CHECKPOINT = None  # None: most recent successful 02 probe; pin a path for A/B comparisons.
config = performance.execution_config(config, PROFILE, stage=2, checkpoint_mode=CHECKPOINT_MODE)
config['training'].update(accumulation=8, validate_every=500, save_every=25)
probe_config = copy.deepcopy(config)
probe_config['training'].update(validate_every=PROBE_STEPS, save_every=PROBE_STEPS)
probe_config.setdefault('context_views', {})['max_candidates'] = 2
if not RUN_FULL_STAGE2:
    STAGE1_CHECKPOINT = Path(STAGE1_CHECKPOINT) if STAGE1_CHECKPOINT else probes.latest_stage1_probe(ROOT)
    probe_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid4().hex[:6]
    PROBE_OUTPUT = ROOT/'runs/stage2_probe'/probe_id/'seed_17'
    print('Stage 1 checkpoint:', STAGE1_CHECKPOINT, flush=True)
    print('Stage 2 output:', PROBE_OUTPUT, flush=True)
    print('Limits:', PROBE_QUESTIONS, 'questions;', PROBE_PAIRS, 'pairs;', PROBE_STEPS, 'updates', flush=True)
else:
    print('FORMAL Stage 2: complete reviews, all reference pairs, formal Stage 1 required.', flush=True)


[execution] a100_40gb: branches <= 4; padded tokens/call <= 1536 (one longer branch remains intact); states=1; effective batch=8; checkpoint=branch. Unmeasured starting configuration: inspect GPU peak and throughput.
Stage 1 checkpoint: /content/drive/MyDrive/SA-Jev/public_lightrag_v1/full/runs/stage1_probe/20261113T082713Z_6e0e78/seed_17/step_000008
Stage 2 output: /content/drive/MyDrive/SA-Jev/public_lightrag_v1/full/runs/stage2_probe/20261113T101744Z_3f6ac1/seed_17
Limits: 16 questions; 8 pairs; 8 updates


In [6]:
if RUN_FULL_STAGE2:
    context_summary = context_views.prepare_context_views(ROOT, config)
else:
    context_summary = probes.prepare_stage2_probe(ROOT, probe_config, PROBE_OUTPUT, STAGE1_CHECKPOINT,
                                                  question_limit=PROBE_QUESTIONS)
context_summary


[stage2 probe] Reviewing at most 16 training questions; no retrieval or trajectory regeneration.
[context] Reading existing tasks and questions; no candidate mining or DPO regeneration.
[context] 16 questions; at most 32 uncached API reviews. Each decision is cached; no model training here.
[context] 1/16; accepted=1 rejected=0 no_source=0; API=0 cached=1; ETA=1s
[context] 2/16; accepted=2 rejected=0 no_source=0; API=0 cached=2; ETA=1s
[context] 3/16; accepted=3 rejected=0 no_source=0; API=0 cached=3; ETA=1s
[context] 4/16; accepted=4 rejected=0 no_source=0; API=0 cached=4; ETA=1s
[context] 5/16; accepted=4 rejected=2 no_source=0; API=0 cached=6; ETA=1s
[context] 6/16; accepted=5 rejected=2 no_source=0; API=0 cached=7; ETA=1s
[context] 7/16; accepted=6 rejected=2 no_source=0; API=0 cached=8; ETA=1s
[context] 8/16; accepted=7 rejected=2 no_source=0; API=0 cached=9; ETA=1s
[context] 9/16; accepted=8 rejected=2 no_source=0; API=0 cached=10; ETA=1s
[context] 10/16; accepted=9 rejected=2 no

{'questions': 16,
 'accepted_questions': 13,
 'eligible_states': 39,
 'eligibility_note': 'before model-token-budget checking; actual pair counts are in training.jsonl',
 'total_states': 36000,
 'rejected_reviews': 7,
 'no_source_questions': 0,
 'new_api_calls': 0,
 'cached_reviews': 20,
 'seconds': 7.678899758000625,
 'review_type': 'LLM-reviewed, not human-verified; semantic errors remain possible'}

In [7]:
assert torch.cuda.is_available(), 'GPU required for training'
if RUN_FULL_STAGE2:
    for seed in TRAIN_SEEDS:
        experiments.train_variant(ROOT, config, 'full', seed)
else:
    history = probes.run_stage2_probe(ROOT, probe_config, PROBE_OUTPUT, STAGE1_CHECKPOINT,
        max_steps=PROBE_STEPS, pair_limit=PROBE_PAIRS, validation_limit=6, stress_check=STRESS_CHECK)
    print('Inspect probe_checks.json, stress_probe.json and throughput.json:', PROBE_OUTPUT, flush=True)
    if (STAGE1_CHECKPOINT.parent/'throughput.json').exists():
        performance.combined_time_report(STAGE1_CHECKPOINT.parent, PROBE_OUTPUT, PROBE_OUTPUT)
    else:
        print('Stage 1 checkpoint has no new throughput report. Run updated 02 for a combined estimate.', flush=True)


[stage2 probe] Repeating the small probe subset for 2 epochs to measure the requested updates; this is not a representative full-data epoch.
[stage2 probe] 48 task states, 8 complete preference pairs, 6 validation states, at most 8 updates; TEST ONLY.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

[OpenJev] Checking artifact: /content/drive/MyDrive/SA-Jev/huggingface/hub/models--shenjunhao--OpenJev-4B/snapshots/064bd435c6713486450194e29d526266b28941e2
[OpenJev] Loading verified ordinary files from local runtime: /tmp/sajev_openjev_models/fd2ba2d05d9eb7ca5bbc3411
[OpenJev] 1/11 backbone/config.json: expected 1,977 bytes; checking SHA-256...
[OpenJev] 2/11 backbone/model-00001-of-00003.safetensors: expected 3,991,295,368 bytes; checking SHA-256...
[OpenJev] 3/11 backbone/model-00002-of-00003.safetensors: expected 3,979,828,128 bytes; checking SHA-256...
[OpenJev] 4/11 backbone/model-00003-of-00003.safetensors: expected 440,425,856 bytes; checking SHA-256...
[OpenJev] 5/11 backbone/model.safetensors.index.json: expected 33,047 bytes; checking SHA-256...
[OpenJev] 6/11 calibration.json: expected 44 bytes; checking SHA-256...
[OpenJev] 7/11 decision_head.safetensors: expected 37,809,208 bytes; checking SHA-256...
[OpenJev] 8/11 openjev_config.json: expected 458 bytes; checking SHA-25

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


[ERROR] `loss` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen3_5/modeling_qwen3_5.py.
[ERROR] `logits` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen3_5/modeling_qwen3_5.py.


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

[OpenJev] Load backbone/head/tokenizer; target=cuda, dtype=bfloat16: exited after 11.7s
[OpenJev] Model loaded on cuda; configuring decision branches
[OpenJev] Delta kernel=fla.ops.gated_delta_rule.chunk.chunk_gated_delta_rule; convolution=torch
[OpenJev] branch_microbatch=4, independent decision batch=8
[OpenJev] optimized_execution=True; branch token budget=1536; checkpoint_mode=branch
[OpenJev] LoRA + decision head trainable parameters: 41,916,416/4,247,667,712 (0.99%); original backbone weights frozen
[stress] 1/2: K=[8], states=1, views=1; backward, NO optimizer update
[stress] {'seconds': 26.68675958899985, 'remaining_k': [8], 'views': 1, 'allocated_gib': 8.099618911743164, 'reserved_gib': 25.41796875, 'peak_allocated_gib': 24.0838303565979, 'peak_reserved_gib': 25.41796875}
[stress] 2/2: K=[20], states=1, views=2; backward, NO optimizer update
[stress] {'seconds': 14.685179997000887, 'remaining_k': [20], 'views': 2, 'allocated_gib': 8.099618911743164, 'reserved_gib': 25.41796875

## 验收与后续
probe_checks.json 必须有 completed_updates=8、dpo_finite=True、checkpoint_saved=True、paths_exercised=True。
比较 task_seconds 与 dpo_seconds；throughput.json 只外推同卡同配置，不凭显存容量推算 H100/A100 80GB 速度。
稳定性按原 100 步升温保留，8 步不能证明最终稳定性或 QA 效果；验证 rank=None 仍表示没有可用金标，不能算排序通过。
一次 OOM 不会自动跳过样本或缩小候选；重启后减少并发／token 分组上限，或测试 branch_layer 重计算模式。
冻结参考与 DPO 仍使用干净完整轨迹，不截断、不改变偏好、不将预算耗尽当作 STOP。新增性能优化不需要重跑 00 或重新生成全部轨迹。
正式实验保留 36,000 状态、所有原有监督、学习率、有效批量和完整轨迹；24 小时是需要实测达成的目标，不以减少数据或省略目标换取。